In [1]:
import pandas as pd

# pubmed_abstract = pd.read_csv('PubMed_200k_RCT_train.csv')
pubmed_abstract_processed = pd.read_csv('pubmed_abstract_cleaned.csv')
# print(pubmed_abstract.shape)
print(pubmed_abstract_processed.shape)

(2211861, 7)


In [2]:
pubmed_abstract_processed.drop(['abstract_id', 'line_id','line_number','abstract_text','target','total_lines'], axis=1, inplace=True)
print(pubmed_abstract_processed.columns.tolist())

['abstract_text_clean']


In [3]:
# import nltk
# import string
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize

# # Download if needed
# nltk.download('stopwords', quiet=True)
# nltk.download('punkt',quiet=True)
# nltk.download('punkt_tab',quiet=True)

# stop_words = set(stopwords.words('english'))

# def clean_text(text):
#     """Clean and tokenize text, remove stop words"""
#     tokens = word_tokenize(str(text).lower())
#     cleaned = []
#     for token in tokens:
#         if token.startswith('-') or token.endswith('-') or token in stop_words:
#             continue
#         if not token and token in stop_words:
#             continue
#         if any(c.isalpha() for c in token):
#             cleaned.append(token)

#     return " ".join(cleaned)

# print('start cleaning...')
# pubmed_abstract_processed['abstract_text_clean'] = pubmed_abstract_processed['abstract_text'].apply(clean_text)
# pubmed_abstract_processed.to_csv('pubmed_abstract_cleaned.csv', index=False)

# print('complete cleaning....')

In [16]:
import time

import re
from collections import Counter

# Save timestamp
start = time.time()

def count_word_occurrences(df, words):
    pct1 = {word: (df['abstract_text_clean'].str.contains(rf'\b{word}\b').sum() / len(df)) * 100 
        for word in target_words}
    print(pd.Series(pct1), end ='\n\n')

    return {word: df['abstract_text_clean'].str.contains(rf'\b{word}\b', regex=True).sum() for word in words}
    

# target_words = ['cloud', 'agent', 'neural', 'stream']
# target_words = ['hallucination', 'platform', 'pipeline']
# target_words = ['model', 'training', "network", "agent" , "bias","neural",'hallucination']

print("period1")
freq1 = count_word_occurrences(pubmed_abstract_processed, target_words)

end = time.time()

print("Took ", end - start)

# Create the DataFrame with a guaranteed index order
comparison = pd.DataFrame(index=target_words)

# Map the dictionaries to that index
comparison['count'] = comparison.index.map(freq1)
print(comparison)

period1
viral    0.148653
virus    0.157695
dtype: float64

Took  16.28466248512268
       count
viral   3288
virus   3488


In [3]:
from gensim.models import Word2Vec

pubmed_tokenized = [sent.split() for sent in pubmed_abstract_processed['abstract_text_clean'] if isinstance(sent, str)]

print"=======================================DO NOT RUN ONLY FOR FRESH SETUP ====================================================")
# print"============================================================================================================================"
print("tokenized complete")

model_pubmed_42 = Word2Vec(pubmed_tokenized, vector_size=200, window=8, min_count=10, sg=1, negative=10, epochs=10, seed=42, workers=1)
model_pubmed_42.save('model_pubmed_200_42.bin')
print("model_pubmed_42 complete")

model_pubmed_123 = Word2Vec(pubmed_tokenized, vector_size=200, window=8, min_count=10, sg=1, negative=10, epochs=10, seed=123, workers=1)
model_pubmed_123.save('model_pubmed_200_123.bin')
print("model_pubmed_123 complete")

model_pubmed_777 = Word2Vec(pubmed_tokenized, vector_size=200, window=8, min_count=10, sg=1, negative=10, epochs=10, seed=777, workers=1)
model_pubmed_777.save('model_pubmed_200_777.bin')
print("model_pubmed_777 complete")

# print"=======================================DO NOT RUN ONLY FOR FRESH SETUP ====================================================")
# print"============================================================================================================================"

tokenized complete
model_pubmed_42 complete
model_pubmed_123 complete
model_pubmed_777 complete


In [4]:
#  ======================================= LOAD SAVED MODELS ============================================

from gensim.models import Word2Vec
# contenders = ['cloud','neural','hallucination','attention']

model_pubmed_42 = Word2Vec.load("./pubmed/model_pubmed_200_42.bin")
model_pubmed_123 = Word2Vec.load("./pubmed/model_pubmed_200_123.bin")
model_pubmed_777 = Word2Vec.load("./pubmed/model_pubmed_200_777.bin")
print("mmodels loaded")

#  ======================================= LOAD SAVED MODELS ============================================

mmodels loaded


In [5]:
def get_neighbours(t_word):
    if t_word in model_pubmed_42.wv:
        print(f'{t_word}\n',model_pubmed_42.wv.most_similar(t_word, topn=10),end='\n')
        print(model_pubmed_123.wv.most_similar(t_word, topn=10),end='\n')
        print(model_pubmed_777.wv.most_similar(t_word, topn=10))
        
        print(model_pubmed_42.wv.get_vecattr(t_word, 'count'),end='\n'*2)
    else:
        print(f"{t_word} ABSENT from vocab")


In [6]:
get_neighbours('cell')
get_neighbours('agent')
get_neighbours('memory')
get_neighbours('neural')
get_neighbours('network')

cell
 [('cells', 0.7118086218833923), ('lymphocyte', 0.6278926134109497), ('cd29', 0.59108966588974), ('k562', 0.5860978960990906), ('hexagonality', 0.5854301452636719), ('lysates', 0.5846347212791443), ('microchimerism', 0.5773332715034485), ('t-suppressor', 0.5746757984161377), ('cd4', 0.5714840888977051), ('cellular', 0.5711938738822937)]
[('cells', 0.699276864528656), ('lymphocyte', 0.6112592220306396), ('k562', 0.6102451086044312), ('cd29', 0.5942298769950867), ('t-suppressor', 0.5907575488090515), ('lysates', 0.590109646320343), ('cellular', 0.58845055103302), ('lymphokine-activated', 0.5720454454421997), ('mononuclear', 0.5701608061790466), ('lymphocytes', 0.5660263299942017)]
[('cells', 0.6913769841194153), ('lymphocyte', 0.6047490835189819), ('k562', 0.5960442423820496), ('cd29', 0.5923722982406616), ('t-suppressor', 0.5809837579727173), ('mononuclear', 0.5757573843002319), ('microchimerism', 0.5732152462005615), ('t-cell', 0.5676912665367126), ('cellular', 0.5669015645980835)

In [22]:
get_neighbours('node')

node
 [('lymph', 0.8922349810600281), ('nodal', 0.8541778922080994), ('nodes', 0.812591016292572), ('sentinel', 0.7281013131141663), ('sln', 0.7136197090148926), ('basins', 0.6925265789031982), ('lymphadenectomy', 0.6893573999404907), ('lymph-node', 0.685774564743042), ('subcarinal', 0.6814985871315002), ('sentinel-node', 0.6803135275840759)]
[('lymph', 0.8889225125312805), ('nodal', 0.8363628387451172), ('nodes', 0.8214045763015747), ('basins', 0.7038594484329224), ('sentinel', 0.6910927891731262), ('subcarinal', 0.6874460577964783), ('sentinel-node', 0.6861476898193359), ('lymphadenectomy', 0.6793830990791321), ('lymph-node', 0.6784471869468689), ('sln', 0.6761441230773926)]
[('lymph', 0.896508514881134), ('nodal', 0.8398863673210144), ('nodes', 0.8297989368438721), ('basins', 0.7259799242019653), ('sentinel', 0.7123159170150757), ('lymph-node', 0.6997243165969849), ('sentinel-node', 0.6960422396659851), ('lymphadenectomy', 0.6890934705734253), ('sln', 0.6877402663230896), ('subcarin

In [14]:
wv = model_pubmed_777.wv
sorted_vocab = sorted(wv.index_to_key, 
                      key=lambda word: wv.get_vecattr(word, "count"), 
                      reverse=True)

tmp = [f"{word}: {wv.get_vecattr(word, 'count')}" for word in sorted_vocab[:100]]
print(', '.join(tmp))

patients: 523909, group: 415855, p: 351740, treatment: 236002, study: 232687, groups: 168972, randomized: 138410, significant: 129390, significantly: 127978, compared: 125641, placebo: 115282, control: 112274, trial: 105203, months: 102272, n: 101668, two: 95569, clinical: 92667, mean: 91219, mg: 88787, therapy: 83617, baseline: 82128, intervention: 79830, effects: 79713, weeks: 79653, time: 77067, pain: 77055, effect: 75768, years: 72384, using: 71354, rate: 70126, days: 68364, received: 67373, subjects: 66481, women: 66293, total: 65821, respectively: 64744, difference: 63824, one: 62942, levels: 62559, efficacy: 62130, blood: 61428, primary: 60629, ci: 60437, risk: 58657, increased: 57591, use: 57124, higher: 56050, may: 55307, randomly: 55151, associated: 54831, differences: 54474, used: 53553, either: 52904, dose: 52781, care: 52677, lower: 51439, surgery: 51266, analysis: 50565, age: 49480, outcome: 49102, three: 48889, results: 48861, children: 47259, disease: 47204, treated: 47